In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import numpy as np
import random

In [2]:
def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [3]:
class PolicyNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.fc1 = nn.Linear(3,16)
        self.fc2 = nn.Linear(16,1)

    def forward(self,x):

        h = F.relu(self.fc1(x))

        return self.fc2(h)

In [4]:
class ValueNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.fc1 = nn.Linear(1,32)
        self.fc2 = nn.Linear(32,1)

    def forward(self,x):

        h = F.relu(self.fc1(x))

        return self.fc2(h)

In [5]:
class TemperatureNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.fc1 = nn.Linear(1,16)
        self.fc2 = nn.Linear(16,1)

    def forward(self,t):

        h = F.relu(self.fc1(t))

        T = F.softplus(self.fc2(h))

        return T

In [6]:
def generate_instance(N):

    weights = torch.rand(N)

    W = 1.0

    return weights, W

In [7]:
def generate_dataset(N, num_instances):

    dataset = []

    for _ in range(num_instances):

        dataset.append(generate_instance(N))

    return dataset

In [8]:
def init_state(N):

    X = torch.zeros(N,N)

    for i in range(N):

        X[i,i] = 1

    return X

In [9]:
def free_capacities(X,weights,W):

    load = torch.matmul(weights,X)

    return W - load

In [10]:
def energy(X):

    used = torch.sum(X,dim=0) > 0

    return torch.sum(used)

In [11]:
def select_item(policy,X,weights,free_caps,T):

    N = X.shape[0]

    bins = torch.argmax(X,dim=1)

    c_bi = free_caps[bins]

    T_vec = torch.full((N,),T)

    features = torch.stack([weights,c_bi,T_vec],dim=1)

    logits = policy(features).squeeze()

    probs = torch.softmax(logits,dim=0)

    dist = torch.distributions.Categorical(probs)

    i = dist.sample()

    return i, dist.log_prob(i)

In [12]:
def select_bin(policy,i,X,weights,free_caps,T):

    N = X.shape[0]

    w_i = weights[i]

    weight_vec = torch.full((N,),w_i)

    T_vec = torch.full((N,),T)

    features = torch.stack([weight_vec,free_caps,T_vec],dim=1)

    logits = policy(features).squeeze()

    mask = free_caps >= w_i

    logits[~mask] = -1e9

    probs = torch.softmax(logits,dim=0)

    dist = torch.distributions.Categorical(probs)

    j = dist.sample()

    return j, dist.log_prob(j)

In [13]:
def apply_move(X,i,j):

    X_new = X.clone()

    old = torch.argmax(X_new[i])

    X_new[i,old] = 0

    X_new[i,j] = 1

    return X_new

In [14]:
def metropolis(X,X_new,T):

    E_old = energy(X)

    E_new = energy(X_new)

    delta = E_new - E_old

    if delta <= 0:

        return X_new

    prob = torch.exp(-delta/T)

    if torch.rand(1) < prob:

        return X_new

    return X

In [15]:
def rollout(X,
            weights,
            W,
            steps,
            item_policy,
            bin_policy,
            temp_net):

    log_probs=[]
    rewards=[]
    states=[]

    for t in range(steps):

        t_tensor = torch.tensor([[t/steps]])

        T = temp_net(t_tensor).item()

        caps = free_capacities(X,weights,W)

        i,logp_i = select_item(item_policy,X,weights,caps,T)

        j,logp_j = select_bin(bin_policy,i,X,weights,caps,T)

        X_new = apply_move(X,i,j)

        X = metropolis(X,X_new,T)

        reward = -energy(X).item()

        log_probs.append(logp_i+logp_j)

        rewards.append(reward)

        states.append(torch.tensor([[energy(X)]],dtype=torch.float32))

    return X,log_probs,rewards,states

In [16]:
def compute_returns(rewards,gamma=0.99):

    returns=[]

    G=0

    for r in reversed(rewards):

        G = r + gamma*G

        returns.insert(0,G)

    return torch.tensor(returns)

In [17]:
def ppo_update(item_policy,
               bin_policy,
               value_net,
               optimizer,
               log_probs,
               rewards,
               states):

    returns = compute_returns(rewards)

    values = torch.cat([value_net(s) for s in states]).squeeze()

    advantages = returns - values.detach()

    log_probs = torch.stack(log_probs)

    policy_loss = -(log_probs * advantages).mean()

    value_loss = (returns - values).pow(2).mean()

    loss = policy_loss + 0.5*value_loss

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    return loss.item()

In [18]:
def train(dataset,
          epochs,
          steps,
          item_policy,
          bin_policy,
          value_net,
          temp_net,
          optimizer):

    best_bins = float("inf")

    for epoch in range(epochs):

        total_bins=0

        for weights,W in dataset:

            X = init_state(len(weights))

            X,log_probs,rewards,states = rollout(
                X,
                weights,
                W,
                steps,
                item_policy,
                bin_policy,
                temp_net
            )

            loss = ppo_update(
                item_policy,
                bin_policy,
                value_net,
                optimizer,
                log_probs,
                rewards,
                states
            )

            bins = energy(X).item()

            total_bins += bins

            best_bins = min(best_bins,bins)

        if epoch%10==0:

            print(
            "epoch",epoch,
            "| avg bins",total_bins/len(dataset),
            "| best",best_bins,
            "| loss",loss
            )

In [19]:
def save_model(item_policy,bin_policy,value_net,temp_net,path):

    torch.save({

    "item":item_policy.state_dict(),
    "bin":bin_policy.state_dict(),
    "value":value_net.state_dict(),
    "temp":temp_net.state_dict()

    },path)

In [22]:
def run_experiment(N):

    seeds=[0,1,2,3,4]

    results=[]

    for seed in seeds:

        set_seed(seed)

        dataset = generate_dataset(N,100)

        item_policy = PolicyNet()
        bin_policy = PolicyNet()
        value_net = ValueNet()
        temp_net = TemperatureNet()

        optimizer = optim.Adam(
        list(item_policy.parameters())+
        list(bin_policy.parameters())+
        list(value_net.parameters())+
        list(temp_net.parameters()),
        lr=1e-3)

        train(dataset,
              epochs=50,
              steps=20,
              item_policy=item_policy,
              bin_policy=bin_policy,
              value_net=value_net,
              temp_net=temp_net,
              optimizer=optimizer)

        save_model(
        item_policy,
        bin_policy,
        value_net,
        temp_net,
        f"model_BIN{N}_seed{seed}.pth")

In [ ]:
run_experiment(50)
run_experiment(100)
run_experiment(200)

epoch 0 | avg bins 40.5 | best 37 | loss 78108.3359375
epoch 10 | avg bins 40.5 | best 34 | loss 20021.861328125
epoch 20 | avg bins 40.41 | best 34 | loss 20478.9453125
epoch 30 | avg bins 40.44 | best 34 | loss 18315.0390625
epoch 40 | avg bins 40.3 | best 34 | loss 20323.166015625
epoch 0 | avg bins 40.0 | best 36 | loss 87900.84375
epoch 10 | avg bins 39.74 | best 35 | loss 19309.203125
epoch 20 | avg bins 40.09 | best 34 | loss 20190.453125
epoch 30 | avg bins 39.88 | best 34 | loss 18999.244140625
epoch 40 | avg bins 40.31 | best 34 | loss 21525.921875
epoch 0 | avg bins 40.06 | best 35 | loss 91516.3203125
epoch 10 | avg bins 40.14 | best 34 | loss 20418.05859375
epoch 20 | avg bins 40.01 | best 34 | loss 22233.087890625
epoch 30 | avg bins 40.3 | best 34 | loss 21228.431640625
epoch 40 | avg bins 40.17 | best 34 | loss 19869.79296875
epoch 0 | avg bins 39.98 | best 37 | loss 97439.984375
epoch 10 | avg bins 40.38 | best 36 | loss 23507.48046875
epoch 20 | avg bins 40.46 | best 